# ROGII Wellbore Geology

## Score: 30.381

In [ ]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
SAMPLE_SUB = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv")

ROLL_WINDOWS = (5, 15, 31)
CORR_WINDOW = 51
CORR_SEARCH_FT = 60
SLOPE_LOOKBACK = 50
LGB_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 80,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100
N_FOLDS = 5

In [ ]:
def list_wells(data_dir: Path) -> list[str]:
    wells = sorted({p.name.split("__")[0] for p in data_dir.glob("*__horizontal_well.csv")})
    return wells


def load_horizontal(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__horizontal_well.csv"
    df = pd.read_csv(path)
    df.insert(0, "well", well)
    df["row_idx"] = np.arange(len(df))
    return df


def load_typewell(well: str, data_dir: Path) -> pd.DataFrame:
    path = data_dir / f"{well}__typewell.csv"
    tw = pd.read_csv(path).sort_values("TVT").reset_index(drop=True)
    return tw


def interp_typewell_gr(tvt_values: np.ndarray, tw: pd.DataFrame) -> np.ndarray:
    tvt = tw["TVT"].to_numpy(dtype=float)
    gr = tw["GR"].to_numpy(dtype=float)
    return np.interp(tvt_values, tvt, gr, left=gr[0], right=gr[-1])


def eval_start_index(df: pd.DataFrame) -> int:
    need_pred = df["TVT_input"].isna()
    if need_pred.any():
        return int(need_pred.idxmax())
    if df["GR"].notna().any():
        return int(df["GR"].notna().idxmax())
    return len(df)


def pre_eval_slope(df: pd.DataFrame, eval_start: int) -> float:
    i0 = max(0, eval_start - SLOPE_LOOKBACK)
    seg = df.iloc[i0:eval_start]
    if len(seg) < 5:
        return 1.0
    if "TVT" in seg.columns and seg["TVT"].notna().sum() >= 5:
        tvt = seg["TVT"].astype(float)
    else:
        tvt = seg["TVT_input"].astype(float)
    md = seg["MD"].astype(float)
    valid = tvt.notna() & md.notna()
    tvt = tvt[valid]
    md = md[valid]
    if len(tvt) < 5:
        return 1.0
    d_md = md.iloc[-1] - md.iloc[0]
    if abs(d_md) < 1e-6:
        return 1.0
    return float(np.clip((tvt.iloc[-1] - tvt.iloc[0]) / d_md, 0.05, 3.0))


def best_corr_tvt(
    gr_win: np.ndarray,
    tw_tvt: np.ndarray,
    tw_gr: np.ndarray,
    center: float,
    slope: float,
) -> tuple[float, float]:
    gr_win = np.asarray(gr_win, dtype=float)
    valid = np.isfinite(gr_win)
    if valid.sum() < 5:
        return center, 0.0
    gw = gr_win.copy()
    gw[~valid] = np.nanmean(gw[valid])
    gw = gw - gw.mean()
    gstd = gw.std()
    if gstd < 1e-6:
        return center, 0.0
    gw = gw / gstd

    step = float(np.median(np.diff(tw_tvt)))
    w = len(gr_win)
    offs = np.arange(w, dtype=float) * slope
    cands = np.arange(center - CORR_SEARCH_FT, center + CORR_SEARCH_FT + step, step)
    if len(cands) == 0:
        return center, 0.0
    tvts = cands[:, None] + offs[None, :]
    segs = np.interp(tvts.ravel(), tw_tvt, tw_gr).reshape(len(cands), w)
    sstd = segs.std(axis=1)
    ok = sstd > 1e-6
    if not ok.any():
        return center, 0.0
    segs_n = (segs[ok] - segs[ok].mean(axis=1, keepdims=True)) / sstd[ok][:, None]
    scores = segs_n @ gw / w
    j = int(np.argmax(scores))
    idx = np.flatnonzero(ok)[j]
    best_t = float(cands[idx] + offs[-1])
    return best_t, float(scores[j])


def add_corr_track(df: pd.DataFrame, tw: pd.DataFrame, slope: float) -> pd.DataFrame:
    tw_tvt = tw["TVT"].to_numpy(dtype=float)
    tw_gr = tw["GR"].to_numpy(dtype=float)
    gr = df["GR"].astype(float).ffill().to_numpy()
    n = len(df)
    tvt_corr = df["tvt_slope"].to_numpy(dtype=float).copy()
    corr_score = np.zeros(n, dtype=float)
    eval_idx = np.where(df["in_eval"].to_numpy())[0]

    for i in eval_idx:
        i0 = max(0, i - CORR_WINDOW + 1)
        gr_win = gr[i0 : i + 1]
        center = df["tvt_slope"].iloc[i]
        tvt_corr[i], corr_score[i] = best_corr_tvt(gr_win, tw_tvt, tw_gr, center, slope)

    df["tvt_corr"] = tvt_corr
    df["corr_score"] = corr_score
    return df


def build_features(h: pd.DataFrame, tw: pd.DataFrame, simulate_eval: bool = False) -> pd.DataFrame:
    df = h.copy()
    eval_start = eval_start_index(df)
    df["eval_start"] = eval_start
    df["in_eval"] = df["row_idx"] >= eval_start
    slope = pre_eval_slope(h, eval_start)

    tvt_in = df["TVT_input"].copy()
    if simulate_eval and not tvt_in.isna().all():
        tvt_in.iloc[eval_start:] = np.nan

    prev_tvt = tvt_in.shift(1)
    df["tvt_anchor"] = prev_tvt.ffill()
    anchor_md = df["MD"].shift(1).where(prev_tvt.notna()).ffill()
    df["md_anchor"] = anchor_md
    df["md_since_anchor"] = df["MD"] - df["md_anchor"]
    df["tvt_slope"] = df["tvt_anchor"] + df["md_since_anchor"] * slope
    df["ft_into_eval"] = np.maximum(0, df["row_idx"] - eval_start)

    df["d_md"] = df["MD"].diff().fillna(0)
    df["d_z"] = df["Z"].diff().fillna(0)

    gr = df["GR"].astype(float)
    gr_filled = gr.ffill()
    for w in ROLL_WINDOWS:
        df[f"gr_roll_mean_{w}"] = gr_filled.rolling(w, min_periods=1).mean()
        df[f"gr_roll_std_{w}"] = gr_filled.rolling(w, min_periods=1).std().fillna(0)
    df["gr_missing"] = gr.isna().astype(np.int8)

    df = add_corr_track(df, tw, slope)

    tw_gr_corr = interp_typewell_gr(df["tvt_corr"].to_numpy(dtype=float), tw)
    df["tw_gr_at_corr"] = tw_gr_corr
    df["gr_minus_tw_corr"] = gr_filled - tw_gr_corr
    df["tvt_corr_minus_slope"] = df["tvt_corr"] - df["tvt_slope"]

    if "TVT" in df.columns:
        df["residual"] = df["TVT"] - df["tvt_corr"]

    return df


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "d_md", "d_z",
    "tvt_anchor", "md_since_anchor", "tvt_slope", "tvt_corr", "tvt_corr_minus_slope",
    "corr_score", "ft_into_eval",
    "GR", "gr_missing",
    "tw_gr_at_corr", "gr_minus_tw_corr",
    *[f"gr_roll_mean_{w}" for w in ROLL_WINDOWS],
    *[f"gr_roll_std_{w}" for w in ROLL_WINDOWS],
]


def featurize_well(well: str, data_dir: Path, simulate_eval: bool = False) -> pd.DataFrame:
    h = load_horizontal(well, data_dir)
    tw = load_typewell(well, data_dir)
    return build_features(h, tw, simulate_eval=simulate_eval)

In [ ]:
def build_training_frame(wells: list[str], data_dir: Path) -> pd.DataFrame:
    parts = []
    for well in wells:
        df = featurize_well(well, data_dir, simulate_eval=True)
        if "TVT" not in df.columns:
            continue
        mask = (
            df["in_eval"]
            & df["GR"].notna()
            & df["TVT"].notna()
            & df["tvt_anchor"].notna()
            & df["tvt_corr"].notna()
            & df["residual"].notna()
        )
        parts.append(df.loc[mask])
    return pd.concat(parts, ignore_index=True)


train_wells = list_wells(TRAIN_DIR)
print(f"Training wells: {len(train_wells)}")

train_df = build_training_frame(train_wells, TRAIN_DIR)
print(f"Training rows (eval zone only): {len(train_df):,}")
print(train_df[["TVT", "tvt_corr", "residual", "corr_score", "GR"]].describe().round(3))

In [ ]:
X = train_df[FEATURE_COLS]
y = train_df["residual"].astype(float)
groups = train_df["well"]

oof_residual = np.zeros(len(train_df))
oof_tvt = np.zeros(len(train_df))
models = []

gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups)):
    dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

    model = lgb.train(
        LGB_PARAMS,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)],
    )
    models.append(model)
    residual_pred = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
    oof_residual[va_idx] = residual_pred
    oof_tvt[va_idx] = train_df.iloc[va_idx]["tvt_corr"].to_numpy() + residual_pred

    fold_rmse = np.sqrt(mean_squared_error(train_df.iloc[va_idx]["TVT"], oof_tvt[va_idx]))
    corr_rmse = np.sqrt(mean_squared_error(train_df.iloc[va_idx]["TVT"], train_df.iloc[va_idx]["tvt_corr"]))
    print(
        f"Fold {fold + 1}/{N_FOLDS}  TVT RMSE = {fold_rmse:.4f}  "
        f"corr-only = {corr_rmse:.4f}  (best_iter={model.best_iteration})"
    )

cv_rmse = np.sqrt(mean_squared_error(train_df["TVT"], oof_tvt))
corr_cv = np.sqrt(mean_squared_error(train_df["TVT"], train_df["tvt_corr"]))
print(f"\nOOF eval-zone TVT RMSE: {cv_rmse:.4f}  |  corr-only: {corr_cv:.4f}")

In [ ]:
def predict_eval_zone(well: str, data_dir: Path, models: list) -> pd.DataFrame:
    h = featurize_well(well, data_dir, simulate_eval=False)
    sub = h.loc[h["in_eval"] & h["TVT_input"].isna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=["id", "tvt"])

    Xp = sub[FEATURE_COLS]
    residual_pred = np.mean([m.predict(Xp, num_iteration=m.best_iteration) for m in models], axis=0)
    sub["tvt"] = sub["tvt_corr"].to_numpy() + residual_pred
    sub["id"] = sub["well"] + "_" + sub["row_idx"].astype(str)
    return sub[["id", "tvt"]]


test_wells = list_wells(TEST_DIR)
print(f"Test wells: {len(test_wells)}")

pred_parts = [predict_eval_zone(w, TEST_DIR, models) for w in test_wells]
submission = pd.concat(pred_parts, ignore_index=True)

sample = pd.read_csv(SAMPLE_SUB)
submission = sample[["id"]].merge(submission, on="id", how="left")
missing = submission["tvt"].isna().sum()
if missing:
    raise ValueError(f"Missing predictions for {missing} sample ids")

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nWrote submission.csv  rows={len(submission):,}")
print(f"tvt range: {submission['tvt'].min():.2f} .. {submission['tvt'].max():.2f}")